<a href="https://colab.research.google.com/github/AjeySrini/AI_Projects/blob/main/Data_Merge_Ajey.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

print("=== TASK 1: Data Preparation ===")

students_data = {
    'student_id': [101, 102, 103, 104, 105, 106, 107],
    'name': ['Alice', 'Bob', None, 'David', 'Emma', 'Frank', 'Grace'],
    'email': ['alice@email.com', 'bob@email.com', 'charlie@email.com', None,
              'emma@email.com', 'frank@email.com', 'grace@email.com'],
    'city': ['Mumbai', 'Delhi', 'Bangalore', 'Mumbai', None, 'Chennai', 'Delhi']
}

students_df = pd.DataFrame(students_data)

enrollments_data = {
    'student_id': [101, 102, 103, 105, 108, 109],
    'course_name': ['Python', 'Data Science', 'Python', 'Machine Learning', 'AI', 'Python'],
    'enrollment_date': ['2024-01-15', '2024-01-20', '2024-02-01',
                        '2024-02-10', '2024-02-15', '2024-03-01']
}

enrollments_df = pd.DataFrame(enrollments_data)

scores_data = {
    'student_id': [101, 102, 104, 105, 106],
    'exam_score': [85, 92, 78, 88, 95]
}

scores_df = pd.DataFrame(scores_data)

print("\nOriginal Students DataFrame:")
print(students_df)
print("\nNull Value Analysis:")
total_rows = len(students_df)

for column in students_df.columns:
    null_count = students_df[column].isnull().sum()
    percentage = (null_count / total_rows) * 100
    print(f"Column: {column}, Nulls: {null_count} ({percentage:.2f}%)")

students_df['city'] = students_df['city'].fillna("Unknown")

students_cleaned = students_df.dropna(subset=['name'])

print("\nCleaned Students DataFrame:")
print(students_cleaned)

=== TASK 1: Data Preparation ===

Original Students DataFrame:
   student_id   name              email       city
0         101  Alice    alice@email.com     Mumbai
1         102    Bob      bob@email.com      Delhi
2         103   None  charlie@email.com  Bangalore
3         104  David               None     Mumbai
4         105   Emma     emma@email.com       None
5         106  Frank    frank@email.com    Chennai
6         107  Grace    grace@email.com      Delhi

Null Value Analysis:
Column: student_id, Nulls: 0 (0.00%)
Column: name, Nulls: 1 (14.29%)
Column: email, Nulls: 1 (14.29%)
Column: city, Nulls: 1 (14.29%)

Cleaned Students DataFrame:
   student_id   name            email     city
0         101  Alice  alice@email.com   Mumbai
1         102    Bob    bob@email.com    Delhi
3         104  David             None   Mumbai
4         105   Emma   emma@email.com  Unknown
5         106  Frank  frank@email.com  Chennai
6         107  Grace  grace@email.com    Delhi


In [2]:
print("\n=== TASK 2: Join Operations ===")

# INNER JOIN
inner_join = pd.merge(students_cleaned, enrollments_df, on='student_id', how='inner')

print("\nInner Join Result:")
print(inner_join)

excluded_students = set(students_cleaned['student_id']) - set(enrollments_df['student_id'])

print("\nExcluded students:", list(excluded_students), "- Not in enrollments table")

# LEFT JOIN
left_join = pd.merge(students_cleaned, enrollments_df, on='student_id', how='left')

print("\nLeft Join Result:")
print(left_join)

null_courses = left_join[left_join['course_name'].isnull()]['student_id'].tolist()

print("\nStudents with null course_name:", null_courses)

# RIGHT JOIN
right_join = pd.merge(students_cleaned, enrollments_df, on='student_id', how='right')

print("\nRight Join Result:")
print(right_join)

missing_names = right_join[right_join['name'].isnull()]['student_id'].tolist()

print("\nStudent IDs without names:", missing_names)

# FULL OUTER JOIN
outer_join = pd.merge(students_cleaned, enrollments_df, on='student_id', how='outer')

print("\nFull Outer Join Result:")
print(outer_join)

missing_data_rows = outer_join[
    outer_join['name'].isnull() | outer_join['course_name'].isnull()
]

print("\nRows with missing name OR course:")
print(missing_data_rows)

# OUTER JOIN WITH INDICATOR
outer_indicator = pd.merge(
    students_cleaned,
    enrollments_df,
    on='student_id',
    how='outer',
    indicator=True
)

print("\nMerge Source Distribution:")
print(outer_indicator['_merge'].value_counts())


=== TASK 2: Join Operations ===

Inner Join Result:
   student_id   name            email     city       course_name  \
0         101  Alice  alice@email.com   Mumbai            Python   
1         102    Bob    bob@email.com    Delhi      Data Science   
2         105   Emma   emma@email.com  Unknown  Machine Learning   

  enrollment_date  
0      2024-01-15  
1      2024-01-20  
2      2024-02-10  

Excluded students: [104, 106, 107] - Not in enrollments table

Left Join Result:
   student_id   name            email     city       course_name  \
0         101  Alice  alice@email.com   Mumbai            Python   
1         102    Bob    bob@email.com    Delhi      Data Science   
2         104  David             None   Mumbai               NaN   
3         105   Emma   emma@email.com  Unknown  Machine Learning   
4         106  Frank  frank@email.com  Chennai               NaN   
5         107  Grace  grace@email.com    Delhi               NaN   

  enrollment_date  
0      2024-01-

In [6]:
print("\n=== TASK 3: Lookup and Automation ===")


score_dict = dict(zip(scores_df['student_id'], scores_df['exam_score']))

students_cleaned.loc[:, 'exam_score'] = students_cleaned['student_id'].map(score_dict)

print("\nLookup Operation Result:")
print(students_cleaned[['student_id', 'name', 'exam_score']])


=== TASK 3: Lookup and Automation ===

Lookup Operation Result:
   student_id   name  exam_score
0         101  Alice        85.0
1         102    Bob        92.0
3         104  David        78.0
4         105   Emma        88.0
5         106  Frank        95.0
6         107  Grace         NaN


In [4]:
def auto_merge(df1, df2, join_type, key_column):

    merged_df = pd.merge(df1, df2, on=key_column, how=join_type)

    result = {
        "result_df": merged_df,
        "row_count": len(merged_df),
        "join_type": join_type
    }

    return result


test_inner = auto_merge(students_cleaned, enrollments_df, "inner", "student_id")

print("\nAutomation Function Test")
print("Join Type:", test_inner["join_type"])
print("Rows in Result:", test_inner["row_count"])
print(test_inner["result_df"].head())


test_left = auto_merge(students_cleaned, enrollments_df, "left", "student_id")

print("\nJoin Type:", test_left["join_type"])
print("Rows in Result:", test_left["row_count"])
print(test_left["result_df"].head())


Automation Function Test
Join Type: inner
Rows in Result: 3
   student_id   name            email     city  exam_score       course_name  \
0         101  Alice  alice@email.com   Mumbai        85.0            Python   
1         102    Bob    bob@email.com    Delhi        92.0      Data Science   
2         105   Emma   emma@email.com  Unknown        88.0  Machine Learning   

  enrollment_date  
0      2024-01-15  
1      2024-01-20  
2      2024-02-10  

Join Type: left
Rows in Result: 6
   student_id   name            email     city  exam_score       course_name  \
0         101  Alice  alice@email.com   Mumbai        85.0            Python   
1         102    Bob    bob@email.com    Delhi        92.0      Data Science   
2         104  David             None   Mumbai        78.0               NaN   
3         105   Emma   emma@email.com  Unknown        88.0  Machine Learning   
4         106  Frank  frank@email.com  Chennai        95.0               NaN   

  enrollment_date  
0 